# KLE Colab Notebook

这个 notebook 基于 `kle` 项目里的 `image_predictor` 流程，面向 Google Colab。

它会完成这些步骤：
- 安装最小依赖
- 拉取或更新 `kle` 仓库
- 检查开奖数据
- 使用 `pytorch_cnn` 训练一个可在 Colab GPU 上运行的模型
- 生成下一期预测与若干注号码
- 对最近若干期做快速回测

建议在 Colab 中把运行时切换到 `GPU`。

In [ ]:
import os
import sys
import subprocess
from pathlib import Path


def run(cmd: str) -> None:
    print(f"+ {cmd}")
    subprocess.run(cmd, shell=True, check=True)


REPO_URL = "https://github.com/jursmatsko/lotto-image-predictor-research.git"
REPO_ROOT = Path("/content/lotto-image-predictor-research")
PROJECT_DIR = REPO_ROOT / "kle"

run("python -m pip install -q pandas matplotlib requests beautifulsoup4 lxml")

try:
    import torch
except ImportError:
    run("python -m pip install -q torch")
    import torch

if not REPO_ROOT.exists():
    run(f"git clone {REPO_URL} {REPO_ROOT}")
else:
    print(f"Repo already exists: {REPO_ROOT}")

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print(f"Python: {sys.version.split()[0]}")
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 数据检查

先确认 `data/data.csv` 能正确读取，并检查最近几期数据结构。

In [ ]:
import pandas as pd


data_path = PROJECT_DIR / "data" / "data.csv"
df = pd.read_csv(data_path)

print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
display(df.head())
display(df.tail())

## 加载应用

这里直接复用仓库里的 `ImagePredictionApp`，但训练模型选择 `pytorch_cnn`，因为它在 Colab 的兼容性和速度都更好。

In [ ]:
from image_predictor.main import ImagePredictionApp
from image_predictor.models.pytorch_image_cnn import DEVICE


app = ImagePredictionApp()
app.loader.load_data()

print(f"Training device: {DEVICE}")
print(app.loader.get_statistics())

## 训练参数

首次运行建议先用较小 epoch 做可运行验证；确认流程正常后再增大训练轮数。

In [ ]:
MODEL_PATH = PROJECT_DIR / "image_predictor" / "models" / "saved" / "colab_pytorch_cnn.pt"
EPOCHS = 20
LEARNING_RATE = 8e-4
NUM_TICKETS = 10
FORCE_RETRAIN = False

print({
    "model_path": str(MODEL_PATH),
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "num_tickets": NUM_TICKETS,
    "force_retrain": FORCE_RETRAIN,
})

In [ ]:
if MODEL_PATH.exists() and not FORCE_RETRAIN:
    print(f"Loading existing model: {MODEL_PATH}")
    app.load_model(str(MODEL_PATH), model_type="pytorch_cnn")
else:
    history = app.train(
        model_type="pytorch_cnn",
        epochs=EPOCHS,
        lr=LEARNING_RATE,
        verbose=True,
    )
    app.save_model(str(MODEL_PATH))

## 预测下一期

这个单元会输出 Top 20 号码、若干注推荐号码，并尝试展示生成的预测热力图。

In [ ]:
from IPython.display import Image, display


top20, tickets, pred_image = app.predict(num_tickets=NUM_TICKETS)
print("Top20:", top20)
print("First 3 tickets:")
for ticket in tickets[:3]:
    print(ticket)

pred_path = PROJECT_DIR / "image_predictor" / "output" / "prediction_next.png"
if pred_path.exists():
    display(Image(filename=str(pred_path)))

## 快速回测

这里不调用项目里默认的 `backtest()`，因为那个分支会走到非 Colab 最优的模型路径。这个单元直接复用当前已经训练好的 `pytorch_cnn` 做最近 10 期的轻量回测。

In [ ]:
import numpy as np


app.loader.load_data()
images = app.loader.create_images().astype(np.float32)
issues = app.loader.issues
encoder = app.encoder
seq_len = app.config.model.sequence_length

eval_start = max(seq_len, len(images) - 10)
rows = []
for test_idx in range(eval_start, len(images)):
    seq = images[test_idx - seq_len:test_idx][np.newaxis, ...]
    pred = app.model.predict(seq)[0]

    pred_nums = set(encoder.decode_single(pred, 20))
    actual_nums = set(encoder.decode_single(images[test_idx], 20))
    hits = len(pred_nums & actual_nums)

    rows.append({
        "issue": issues[test_idx],
        "hits": hits,
        "predicted": sorted(pred_nums),
        "actual": sorted(actual_nums),
    })

result_df = pd.DataFrame(rows)
display(result_df[["issue", "hits"]])
print(f"Average hits: {result_df['hits'].mean():.2f}/20")
print("Random baseline: 5.00/20")

## 可选：保存到 Google Drive

如果你想长期保留模型文件，可以在 Colab 中挂载 Drive 后，把 `MODEL_PATH` 改到你的 Drive 目录，例如：

```python
from google.colab import drive
drive.mount('/content/drive')
MODEL_PATH = Path('/content/drive/MyDrive/kle_models/colab_pytorch_cnn.pt')
```

之后重新运行训练单元即可。